# Phase 2-4 経路2: 宅建士過去問 OCR パイプライン (2016-2022, 9PDF)

本ノートは画像PDF (CCITT Fax) で配布されている 2016〜2022 年度 (9ファイル) の宅建士過去問を OCR して、経路1 (`pdfplumber` 抽出) と同一スキーマの中間 JSON に変換します。

## 前提
- Google Drive をマウント済み
- `/content/drive/MyDrive/construction-llm-ft/data/raw/takken/<year>/*.pdf` に PDF が配置されている (git pull で同期)
- `data/raw/takken/manifest.json` も Drive 上に存在

## 出力
- `/content/drive/MyDrive/construction-llm-ft/data/jsonl/intermediate/<year>[_<session>].json`
- 経路1 (`scripts/extract_pdf_text.py`) と完全同一スキーマ。仕様は `docs/freeze-spec.md` §1 を参照

## 想定実行時間
- 9PDF × 約 50問、Colab CPU で OCR。1PDF あたり 3〜5 分、合計 30〜45 分目安

## 依存パッケージ
- `poppler-utils` (`pdftoppm`)、`tesseract-ocr`、`tesseract-ocr-jpn`
- Python: `pytesseract`, `pillow`, `tqdm`

## 対象 9PDF
- 2016, 2017, 2018, 2019 (各1)
- 2020-oct, 2020-dec (2)
- 2021-oct, 2021-dec (2)
- 2022 (1)

## 1. システム依存パッケージのインストール

`poppler-utils` は `pdftoppm`、`tesseract-ocr-jpn` は日本語学習データを提供する。

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-jpn
!tesseract --version | head -3
!pdftoppm -v 2>&1 | head -3

## 2. Python パッケージのインストール

In [ ]:
!pip install -q pytesseract pillow tqdm

## 3. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 作業ディレクトリと manifest.json 読み込み

In [ ]:
import json
import os
from pathlib import Path

WORK_DIR = Path('/content/drive/MyDrive/construction-llm-ft')
RAW_DIR = WORK_DIR / 'data' / 'raw' / 'takken'
OUT_DIR = WORK_DIR / 'data' / 'jsonl' / 'intermediate'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RAW_DIR / 'manifest.json'
assert MANIFEST_PATH.exists(), f'manifest.json が見つかりません: {MANIFEST_PATH}'

with MANIFEST_PATH.open('r', encoding='utf-8') as f:
    manifest = json.load(f)

# OCR 対象は extractability == 'image_pdf_needs_ocr' のもの (= 2016-2022 の9件)
OCR_TARGETS = [item for item in manifest['items'] if item.get('extractability') == 'image_pdf_needs_ocr']
print(f'OCR対象: {len(OCR_TARGETS)} 件')
for item in OCR_TARGETS:
    print(f"  {item['year']}{('-' + item['session']) if item['session'] else ''}: {item['local_path']}")

## 5. PDF→PNG 変換ヘルパ

`pdftoppm -r 300 -png` で各ページを PNG 化し、ソートされたパスのリストを返す。一時ディレクトリは呼び出し側で管理。

In [ ]:
import subprocess
import tempfile
from pathlib import Path
from typing import List

def pdf_to_pngs(pdf_path: Path, out_dir: Path, dpi: int = 300) -> List[Path]:
    """PDF を 1 ページごとの PNG に変換し、ソート済みのパスリストを返す。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = out_dir / 'page'
    cmd = ['pdftoppm', '-r', str(dpi), '-png', str(pdf_path), str(prefix)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'pdftoppm 失敗: {result.stderr}')
    pngs = sorted(out_dir.glob('page-*.png'))
    if not pngs:
        raise RuntimeError(f'PNG 生成なし: {pdf_path}')
    return pngs

## 6. OCR ヘルパ

`pytesseract.image_to_string` で日本語 OCR。`--psm 6` (均一テキストブロック想定) を使用。各ページの結果を改ページマーカで連結する。

In [ ]:
import pytesseract
from PIL import Image
from typing import List, Tuple

TESSERACT_CONFIG = '--psm 6'


def ocr_pages(png_paths: List[Path]) -> Tuple[str, List[str]]:
    """複数ページ PNG を OCR して、結合済みテキストとページごとテキストを返す。"""
    page_texts: List[str] = []
    for p in png_paths:
        img = Image.open(p)
        txt = pytesseract.image_to_string(img, lang='jpn', config=TESSERACT_CONFIG)
        page_texts.append(txt)
    combined = '\n\n=== PAGE BREAK ===\n\n'.join(page_texts)
    return combined, page_texts


def get_tesseract_version() -> str:
    try:
        v = pytesseract.get_tesseract_version()
        return f'tesseract-jpn-v{v}'
    except Exception:
        return 'tesseract-jpn-vunknown'

## 7. 文字正規化と問題パース関数

経路1 (`scripts/extract_pdf_text.py`) と同等のパースロジックをノート内に再定義する (経路1未完成のため重複OK)。

OCR 出力は誤認識・行ずれが多いので、以下の方針:
- 全角数字 (`１２３４`) → 半角に正規化
- 「問1」「問１」「問 1」など複数の問題開始記法に対応
- 選択肢は `1`/`2`/`3`/`4` で始まる行を検出 (前置記号「(1)」「①」も許容)
- 末尾に出てくる正答一覧 (例: `1 3 / 2 2 / 3 4 ...` のような表) は別途パース
- 上記で取れない問題は `correct_answer = None` を許容 (後段の build_jsonl.py で警告)
- 50問取り切れない場合も実行は継続し、最終検証セルで警告を出す

In [ ]:
import re
import unicodedata
from typing import Dict, List, Optional

ZEN_TO_HAN_DIGITS = str.maketrans('０１２３４５６７８９', '0123456789')
MARU_TO_DIGIT = {'①': '1', '②': '2', '③': '3', '④': '4'}


def normalize_text(s: str) -> str:
    """OCR 出力の基本正規化: NFKC + 全角数字→半角 + ① 系 → 半角."""
    if s is None:
        return ''
    s = unicodedata.normalize('NFKC', s)
    s = s.translate(ZEN_TO_HAN_DIGITS)
    for k, v in MARU_TO_DIGIT.items():
        s = s.replace(k, v)
    # 連続空白を 1 つにまとめるが、改行は保持
    lines = []
    for line in s.split('\n'):
        line = re.sub(r'[ \t\u3000]+', ' ', line).strip()
        lines.append(line)
    return '\n'.join(lines)


QUESTION_HEAD_RE = re.compile(r'^\s*(?:【\s*)?問\s*(\d{1,2})(?:\s*】)?\s*[\.。、:：]?\s*(.*)$')
CHOICE_HEAD_RE = re.compile(r'^\s*\(?(?:[\(（]?\s*([1-4])\s*[\)）]?)\s*[\.。、:：]?\s*(.+)$')


def split_into_questions(full_text: str) -> List[Dict]:
    """OCR テキスト全体から「問N」ブロックを切り出して 50 問ぶん収集する。

    アルゴリズム:
    1. ページ区切りを除去して 1 つのテキストにする
    2. 「問N」見出しの位置をすべて検出
    3. 隣り合う見出し間のテキストをその問題の raw_block にする
    4. raw_block 内の「1〜4」始まりの行を選択肢に割り当てる (最初の出現を採用)
    """
    text = normalize_text(full_text)
    text_no_break = text.replace('=== PAGE BREAK ===', '\n')

    # 「問1」〜「問50」の見出し位置を全探索 (行頭でなくてもOKだが、直前が改行か文頭であること)
    head_pattern = re.compile(r'(?:^|\n)\s*(?:【\s*)?問\s*(\d{1,2})(?:\s*】)?\s*', re.MULTILINE)
    heads = [(m.start(), m.end(), int(m.group(1))) for m in head_pattern.finditer(text_no_break)]
    # 同じ番号が複数回出ても最初の出現を使う
    seen = set()
    unique_heads = []
    for start, end, num in heads:
        if num in seen or num < 1 or num > 50:
            continue
        seen.add(num)
        unique_heads.append((start, end, num))
    unique_heads.sort(key=lambda x: x[0])

    questions = []
    for i, (start, end, num) in enumerate(unique_heads):
        block_end = unique_heads[i + 1][0] if i + 1 < len(unique_heads) else len(text_no_break)
        raw_block = text_no_break[start:block_end].strip()
        parsed = parse_question_block(raw_block, num)
        if parsed is not None:
            questions.append(parsed)
    return questions


def parse_question_block(block: str, number: int) -> Optional[Dict]:
    """問題ブロック 1 つから question_text / choices / raw_block を抽出."""
    lines = [l for l in block.split('\n') if l.strip()]
    if not lines:
        return None

    # 1 行目の「問N」を除去
    head_re = re.compile(r'^\s*(?:【\s*)?問\s*\d{1,2}(?:\s*】)?\s*[\.。、:：]?\s*')
    first_line = head_re.sub('', lines[0]).strip()

    # 選択肢の開始行を検出
    choice_re = re.compile(r'^\s*([1-4])\s*[\.。、:：\)）]\s*(.+)$')
    choices: Dict[str, str] = {}
    question_lines: List[str] = []
    if first_line:
        question_lines.append(first_line)

    current_choice: Optional[str] = None
    current_choice_buf: List[str] = []

    def flush_choice():
        if current_choice is not None and current_choice not in choices:
            choices[current_choice] = ' '.join(current_choice_buf).strip()

    for line in lines[1:]:
        m = choice_re.match(line)
        if m:
            # 直前の選択肢を確定
            flush_choice()
            current_choice = m.group(1)
            current_choice_buf = [m.group(2).strip()]
        else:
            if current_choice is None:
                question_lines.append(line.strip())
            else:
                current_choice_buf.append(line.strip())
    flush_choice()

    question_text = ' '.join(question_lines).strip()

    # 全選択肢が揃っていなくても出力 (空文字で埋める)。後段で警告
    full_choices = {str(k): choices.get(str(k), '') for k in (1, 2, 3, 4)}

    return {
        'number': number,
        'question_text': question_text,
        'choices': full_choices,
        'raw_block': block,
    }


def parse_answer_key(full_text: str) -> Dict[int, str]:
    """PDF 末尾の正答一覧から {問番号: '1'..'4'} の dict を作る。

    RETIO の宅建PDF では、巻末に「正解番号」「解答」表があり、
    `1 3 / 2 2 / 3 4 ...` のように「問番号 + 正答」が並ぶ。
    OCR 揺らぎが大きいので寛容にマッチさせる。
    """
    text = normalize_text(full_text)
    answers: Dict[int, str] = {}

    # 戦略 1: 「問 N N」または「N N」のペアを巻末方向から拾う
    # 末尾 3000 文字を対象に絞る (誤検出回避)
    tail = text[-6000:] if len(text) > 6000 else text

    # 「問 12 3」形式 (問番号 + 正答)
    pat1 = re.compile(r'問\s*(\d{1,2})\s*[\.。、:：]?\s*([1-4])')
    for m in pat1.finditer(tail):
        n = int(m.group(1))
        a = m.group(2)
        if 1 <= n <= 50 and n not in answers:
            answers[n] = a

    if len(answers) >= 40:
        return answers

    # 戦略 2: 表組の数字列をペアで読む (1行に「問 N」と次列「答 X」が分かれているケース)
    pat2 = re.compile(r'(?:^|\s)(\d{1,2})\s+([1-4])(?=\s|$)', re.MULTILINE)
    for m in pat2.finditer(tail):
        n = int(m.group(1))
        a = m.group(2)
        if 1 <= n <= 50 and n not in answers:
            answers[n] = a

    return answers

## 8. ID 生成と中間 JSON ビルダ

ID 命名は `docs/freeze-spec.md` §1 を厳守: `takken_<year>[_<session>]_q<NNN>` (3 桁ゼロ埋め)。

In [ ]:
from datetime import datetime, timezone, timedelta

JST = timezone(timedelta(hours=9))


def make_question_id(year: int, session: Optional[str], number: int) -> str:
    parts = ['takken', str(year)]
    if session:
        parts.append(session)
    parts.append(f'q{number:03d}')
    return '_'.join(parts)


def build_intermediate_json(item: Dict, full_text: str, extractor_label: str) -> Dict:
    """manifest item と OCR テキストから中間 JSON dict を作る (経路1と同一スキーマ)."""
    year = item['year']
    session = item.get('session')
    questions = split_into_questions(full_text)
    answer_key = parse_answer_key(full_text)

    out_questions = []
    for q in questions:
        num = q['number']
        out_questions.append({
            'id': make_question_id(year, session, num),
            'number': num,
            'question_text': q['question_text'],
            'choices': q['choices'],
            'correct_answer': answer_key.get(num),
            'raw_block': q['raw_block'],
        })
    out_questions.sort(key=lambda x: x['number'])

    return {
        'exam_code': 'takken',
        'year': year,
        'session': session,
        'source_pdf': item['local_path'],
        'extracted_at': datetime.now(JST).isoformat(),
        'extractor': extractor_label,
        'questions': out_questions,
    }


def output_path_for(item: Dict) -> Path:
    year = item['year']
    session = item.get('session')
    name = f'{year}_{session}.json' if session else f'{year}.json'
    return OUT_DIR / name

## 9. メイン実行ループ

manifest の OCR 対象 9 件を順に処理。`overwrite=False` なら既存 JSON はスキップ。

In [ ]:
from tqdm.auto import tqdm

OVERWRITE = False  # 既存 JSON を再生成したい場合は True に

EXTRACTOR_LABEL = get_tesseract_version()
print(f'Extractor: {EXTRACTOR_LABEL}')

results_summary = []

for item in tqdm(OCR_TARGETS, desc='OCR PDFs'):
    out_path = output_path_for(item)
    if out_path.exists() and not OVERWRITE:
        print(f'[skip] {out_path.name} は既に存在')
        with out_path.open('r', encoding='utf-8') as f:
            existing = json.load(f)
        results_summary.append({
            'year': item['year'],
            'session': item.get('session'),
            'num_questions': len(existing.get('questions', [])),
            'skipped': True,
        })
        continue

    pdf_path = WORK_DIR / item['local_path']
    if not pdf_path.exists():
        print(f'[error] PDF が存在しません: {pdf_path}')
        continue

    label = f"{item['year']}{('-' + item['session']) if item.get('session') else ''}"
    print(f'\n=== {label} ({pdf_path.name}) ===')

    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir)
        pngs = pdf_to_pngs(pdf_path, tmp_path, dpi=300)
        print(f'  PNG ページ数: {len(pngs)}')
        full_text, _ = ocr_pages(pngs)

    intermediate = build_intermediate_json(item, full_text, EXTRACTOR_LABEL)
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(intermediate, f, ensure_ascii=False, indent=2)
    print(f'  -> 保存: {out_path} ({len(intermediate["questions"])} 問)')

    results_summary.append({
        'year': item['year'],
        'session': item.get('session'),
        'num_questions': len(intermediate['questions']),
        'num_with_answer': sum(1 for q in intermediate['questions'] if q['correct_answer']),
        'skipped': False,
    })

print('\n=== 全件完了 ===')
for r in results_summary:
    print(r)

## 10. 検証セル

出力された各中間 JSON について以下をチェック:
- `len(questions) == 50`
- 各問の `correct_answer` が `'1'..'4'` のいずれか
- 各問の `choices` 4 つが非空

未達は警告として出力 (実行は止めない)。OCR 揺らぎでパース漏れが出ることはあり得るので、後続 (Phase 2-5 `build_jsonl.py`) で再正規化・人手補正する想定。

In [ ]:
from typing import List

VALID_ANSWERS = {'1', '2', '3', '4'}

warnings: List[str] = []

for item in OCR_TARGETS:
    out_path = output_path_for(item)
    if not out_path.exists():
        warnings.append(f'[missing] {out_path.name} が存在しない')
        continue
    with out_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    label = f"{data['year']}{('-' + data['session']) if data.get('session') else ''}"
    qs = data.get('questions', [])

    if len(qs) != 50:
        warnings.append(f'[{label}] 問題数 {len(qs)} (期待 50)')

    bad_answers = [q['number'] for q in qs if q.get('correct_answer') not in VALID_ANSWERS]
    if bad_answers:
        warnings.append(f'[{label}] correct_answer 異常: 問{bad_answers}')

    bad_choices = []
    for q in qs:
        ch = q.get('choices', {})
        for k in ('1', '2', '3', '4'):
            if not ch.get(k):
                bad_choices.append((q['number'], k))
    if bad_choices:
        warnings.append(f'[{label}] 空の選択肢: {bad_choices[:10]}{" ..." if len(bad_choices) > 10 else ""} (計 {len(bad_choices)})')

if warnings:
    print(f'警告 {len(warnings)} 件:')
    for w in warnings:
        print(' -', w)
else:
    print('全 9 ファイル、全問正常 (50問・正答・選択肢4つ揃い)')

## 11. クロージング

- 中間 JSON は `/content/drive/MyDrive/construction-llm-ft/data/jsonl/intermediate/` に保存されました
- `data/jsonl/**/*.jsonl` は `.gitignore` で除外されていますが、中間 JSON (`*.json`) は除外対象に含まれていない場合 commit されます。著作権配慮のためコミット前に `.gitignore` を確認してください
- Mac Mini 側で `git pull` (または rclone/手動コピー) を実行すれば中間 JSON を取り込めます
- 次は Phase 2-5: `scripts/build_jsonl.py` で freeze-spec.md 準拠の `{instruction, input, output}` 形式へ整形 → train/eval 分割

### OCR 精度が低い問題への対応 (必要に応じて手動)
- 警告セルで「空の選択肢」「correct_answer 異常」が出た年度は、対応する `<year>[_<session>].json` を直接開いて該当 `raw_block` を確認し、`question_text` / `choices` / `correct_answer` を手動修正する
- 修正後の JSON を再度 Drive に保存しておけば、Mac Mini 側で `git pull` (または手動コピー) で反映可能